# 005 · Sorted Sequences with bisect

> **Standard Library › Data Structures** · Python 3.14

`bisect` provides binary search over an **already-sorted** list: find where a value is (or would go) in O(log n), and insert while keeping the list sorted. It's the building block for range lookups, threshold tables and simple sorted collections.

**In this notebook you will learn:**

- `bisect_left` vs `bisect_right` and how they relate to `List<T>.BinarySearch`
- how to test membership and count values in a range
- keeping a list sorted with `insort`
- the `key=` parameter (3.10+) and its asymmetry between `bisect_*` and `insort_*`
- the classic "grade lookup" table, and what to use instead of `SortedList` / `SortedSet`

## .NET mental model

| .NET | Python |
|---|---|
| `list.BinarySearch(x)` → index, or `~insertionPoint` if missing | `bisect.bisect_left(lst, x)` → always the insertion point |
| `Array.BinarySearch(arr, x)` | same as above |
| `list.Insert(~list.BinarySearch(x), x)` | `bisect.insort(lst, x)` |
| `BinarySearch(x, IComparer<T>)` | `bisect_left(lst, k, key=f)` (3.10+) |
| `SortedList<K,V>`, `SortedSet<T>`, `SortedDictionary<K,V>` | **no stdlib equivalent** — sorted `list` + `bisect`, or the third-party `sortedcontainers` package |

## `bisect_left` and `bisect_right`

Both return an **insertion point**: the index where `x` could be inserted to keep the list sorted. They only differ when `x` is already present:

- `bisect_left` returns the position **before** any existing equal items (the first index with `a[i] >= x`)
- `bisect_right` (alias `bisect.bisect`) returns the position **after** them (the first index with `a[i] > x`)

In [ ]:
import bisect

a = [10, 20, 20, 20, 30, 40]
#    0   1   2   3   4   5

print(bisect.bisect_left(a, 20))    # 1 — before the 20s
print(bisect.bisect_right(a, 20))   # 4 — after the 20s
print(bisect.bisect(a, 20))         # 4 — bisect is bisect_right
print(bisect.bisect_left(a, 25))    # 4 — not present: both agree
print(bisect.bisect_left(a, 5), bisect.bisect_left(a, 99))   # 0 and len(a)

## Compared with `BinarySearch`

.NET's `BinarySearch` returns the index of **a** matching element when found, or the bitwise complement (`~`) of the insertion point when not. Python's `bisect` functions never tell you directly whether the value was found — you check the element at the returned index yourself.

```csharp
// C#
var list = new List<int> { 10, 20, 30, 40 };
int i = list.BinarySearch(25);          // -4 (i.e. ~3)
if (i < 0) list.Insert(~i, 25);         // insert at 3
bool found = list.BinarySearch(30) >= 0;
```

In [ ]:
def contains(a: list, x) -> bool:
    i = bisect.bisect_left(a, x)
    return i < len(a) and a[i] == x

def index_of(a: list, x) -> int:
    """Like list.index, but O(log n) on a sorted list; -1 if missing."""
    i = bisect.bisect_left(a, x)
    return i if i < len(a) and a[i] == x else -1

a = [10, 20, 30, 40]
print(contains(a, 30), contains(a, 25))
print(index_of(a, 40), index_of(a, 5))

**Tip:** `bisect_left` always finds the **first** of several equal elements; .NET's `BinarySearch` may return *any* of them. That makes `bisect` handy for range queries — count how many values fall in `[lo, hi]` with two searches:

In [ ]:
temps = sorted([18.5, 21.0, 19.2, 25.3, 22.8, 20.1, 23.4, 21.0])

def count_between(a, lo, hi):
    return bisect.bisect_right(a, hi) - bisect.bisect_left(a, lo)

print(temps)
print(count_between(temps, 20.0, 23.0))     # 20.1, 21.0, 21.0, 22.8
print(count_between(temps, 21.0, 21.0))     # exactly 21.0 -> 2

## Keeping a list sorted: `insort`

`insort(a, x)` (alias of `insort_right`) finds the position with a binary search and inserts `x` there; `insort_left` inserts before equal items.

In [ ]:
scores = []
for s in [72, 95, 60, 88, 72]:
    bisect.insort(scores, s)
    print(scores)

⚠️ **Gotcha:** the *search* is O(log n), but the *insert* is O(n) because `list.insert` shifts elements — the same as `List<T>.Insert`. Fine for thousands of items; for heavy insert/delete workloads on big collections a balanced tree (`SortedSet<T>` in .NET; `sortedcontainers.SortedList` in Python) is better. If you only need the smallest item, use a heap *(see Standard Library › Data Structures › 004 · Priority Queues with heapq)*.

⚠️ **Gotcha:** `bisect` assumes the list **is sorted** and doesn't check. On unsorted data it returns meaningless positions without any error.

In [ ]:
unsorted = [30, 10, 40, 20]
print(bisect.bisect_left(unsorted, 20))   # a "valid-looking" but meaningless answer

## The grade-lookup pattern

`bisect` turns a table of numeric **breakpoints** into a lookup function — no chain of `if/elif`. This is the canonical example from the Python docs.

```csharp
// C#
static char Grade(int score) => score switch
{
    >= 90 => 'A', >= 80 => 'B', >= 70 => 'C', >= 60 => 'D', _ => 'F'
};
```

In [ ]:
def grade(score: int, breakpoints=(60, 70, 80, 90), grades="FDCBA") -> str:
    return grades[bisect.bisect(breakpoints, score)]

print([grade(s) for s in [33, 60, 69, 70, 89, 90, 100]])

The same idea works for tax brackets, shipping-cost tiers, or mapping timestamps to time buckets — keep the breakpoints in one sorted sequence and the results in another.

In [ ]:
weight_limits = [1.0, 5.0, 20.0]                   # kg
costs = ["small", "medium", "large", "freight"]

for w in [0.4, 1.0, 3.2, 25.0]:
    print(w, "->", costs[bisect.bisect_left(weight_limits, w)])

## The `key=` parameter (3.10+)

Since Python 3.10 all `bisect` and `insort` functions accept `key=`, so you can search a list of records sorted by a field.

⚠️ **Gotcha:** the key function is applied differently:

- `bisect_left(a, x, key=f)` — `x` is already a **key value**; `f` is applied only to the list elements
- `insort(a, x, key=f)` — `x` is a **whole item**; `f` is applied to `x` as well

In [ ]:
from operator import itemgetter

events = [("boot", 1), ("login", 5), ("upload", 9), ("logout", 14)]   # sorted by time
by_time = itemgetter(1)

i = bisect.bisect_left(events, 9, key=by_time)        # pass the KEY (a time)
print(i, events[i])

bisect.insort(events, ("sync", 7), key=by_time)       # pass the ITEM
print(events)

**Tip:** for older Pythons (or many lookups), keep a parallel list of keys: `keys = [e[1] for e in events]` and bisect into that.

## Restricting the search: `lo` and `hi`

All functions take optional `lo` / `hi` bounds, like the `index`/`count` overload of `List<T>.BinarySearch(index, count, item, comparer)`. Useful when you already know the answer lies in part of the list — e.g. searching only *after* a previous hit.

In [ ]:
a = [1, 3, 3, 3, 7, 9]
first = bisect.bisect_left(a, 3)
print(first, bisect.bisect_left(a, 7, lo=first))   # second search starts at `first`

## A minimal sorted collection

Python has no `SortedSet<T>`/`SortedList<T>` in the standard library, but a thin wrapper over a list and `bisect` covers many everyday needs:

```csharp
// C#
var set = new SortedSet<int> { 5, 1, 3 };
var view = set.GetViewBetween(2, 5);   // 3, 5
```

In [ ]:
class SortedBag:
    """A sorted multiset backed by a list (O(log n) search, O(n) insert)."""

    def __init__(self, items=()):
        self._items = sorted(items)

    def add(self, x) -> None:
        bisect.insort(self._items, x)

    def __contains__(self, x) -> bool:
        i = bisect.bisect_left(self._items, x)
        return i < len(self._items) and self._items[i] == x

    def between(self, lo, hi) -> list:
        """Items with lo <= item <= hi (like GetViewBetween, but a copy)."""
        return self._items[bisect.bisect_left(self._items, lo):bisect.bisect_right(self._items, hi)]

    def __repr__(self) -> str:
        return f"SortedBag({self._items})"

bag = SortedBag([5, 1, 3])
bag.add(4)
print(bag, 3 in bag, 2 in bag)
print(bag.between(2, 5))

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Binary search | `list.BinarySearch(x)` | `bisect.bisect_left(a, x)` |
| Found? | `result >= 0` | `i < len(a) and a[i] == x` |
| Insertion point | `~result` | the return value itself |
| First of duplicates | not guaranteed | `bisect_left` |
| After duplicates | — | `bisect_right` / `bisect` |
| Insert keeping order | `Insert(~BinarySearch(x), x)` | `bisect.insort(a, x)` |
| Search by field | `BinarySearch(x, comparer)` | `bisect_left(a, k, key=f)` |
| Threshold table | `switch` with relational patterns | `grades[bisect(breakpoints, score)]` |
| Sorted collection type | `SortedList` / `SortedSet` | sorted `list` + `bisect` (or `sortedcontainers`) |

## Exercises

1. Write `age_group(age)` that maps ages to `"child"` (< 13), `"teen"` (13–17), `"adult"` (18–64) and `"senior"` (65+) using `bisect`.
2. Given a sorted list of order timestamps (ints), count how many orders fall between 100 and 200 inclusive.
3. Maintain a sorted leaderboard of `(name, score)` tuples ordered by score with `insort` and `key=`, inserting `("alice", 50)`, `("bob", 80)`, `("carol", 65)`.
4. Find the closest value to `x = 23` in the sorted list `[3, 10, 21, 26, 40]`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import bisect

# 1 — breakpoints are the first age of each next group
def age_group(age: int) -> str:
    return ["child", "teen", "adult", "senior"][bisect.bisect([13, 18, 65], age)]

print([age_group(a) for a in [5, 13, 17, 18, 64, 65]])

# 2
times = [50, 99, 100, 150, 180, 200, 201, 300]
print(bisect.bisect_right(times, 200) - bisect.bisect_left(times, 100))   # 4

# 3
board = []
for entry in [("alice", 50), ("bob", 80), ("carol", 65)]:
    bisect.insort(board, entry, key=lambda e: e[1])
print(board)

# 4 — the closest is either just before or at the insertion point
def closest(a, x):
    i = bisect.bisect_left(a, x)
    candidates = a[max(i - 1, 0): i + 1]
    return min(candidates, key=lambda v: abs(v - x))

print(closest([3, 10, 21, 26, 40], 23))
```

</details>